# UT 1 · Práctica 2 — Del área intermedia al *data lake* (HDFS)
**Módulo:** Sistemas de Big Data (SPS · 5074) · CE en Inteligencia Artificial y Big Data

**Relación con el módulo:** RA 3 · *Gestiona y almacena datos facilitando la búsqueda de respuestas en grandes conjuntos de datos.*
- **a)** Se han extraído y almacenado datos de diversas fuentes, para ser tratados en distintos escenarios. *(criterio principal)*
- **b)** Se ha fijado el objetivo de extraer valor de los datos para lo que es necesario contar con tecnologías eficientes. *(en parte)*
- **c)** Se ha comprobado que la revolución digital exige poder almacenar y procesar ingentes cantidades de datos de distinto tipo y descubrir su valor. *(en parte)*
- **d)** Se han desarrollado sistemas de gestión, almacenamiento y procesamiento de grandes volúmenes de datos de manera eficiente y segura, teniendo en cuenta la normativa existente. *(en parte)*

---

### El tema: el alojamiento turístico en Canarias, ahora en un *data lake*

Vas a llevar al *data lake* todo lo que guardaste en el área intermedia en la práctica 1, más dos documentos del ISTAC:

| Origen | Qué archivos son | Formato |
|---|---|---|
| Práctica 1, fuentes 1 y 2 (Gobierno de Canarias) | Hoteles; viviendas vacacionales (JSON original y tabla) | CSV o Parquet, JSON |
| Práctica 1, fuente 3 (ISTAC) | Ocupación hotelera | CSV y Parquet |
| Práctica 1, fuente 4 (cadena hotelera simulada) | Tablas de la base de datos y extracción selectiva | CSV o Parquet |
| Práctica 1 | `registro_extracciones.csv` | CSV |
| Aula virtual (ISTAC) | Metodología de la encuesta y aviso legal | PDF |

### ¿Dónde estamos dentro de una arquitectura de datos?

```
 ÁREA INTERMEDIA (staging/)         CARGA                     DATA LAKE EN HDFS (/datalake)
 ──────────────────────────         ─────────────────         ─────────────────────────────
 archivos de la práctica 1   ──┐                              bruto/    ◄── esta práctica
 registro de extracciones    ──┼──► cargar_en_bruto()  ──►    limpio/   (vacía)
 PDF del ISTAC               ──┘    sin modificar ni          consumo/  (vacía)
                                    sobrescribir
```

En un *data lake* los datos se guardan **tal como vienen** y se interpretan después, al leerlos (*schema-on-read*). Por eso aquí **no limpiamos ni transformamos nada**: copiamos los archivos del área intermedia a la **zona en bruto** del lago.
HDFS es una **caja negra**: el entorno está preparado, no instalas ni configuras Hadoop, solo lo usas.

## Cómo usar este cuaderno

- Un cuaderno tiene **celdas de texto** (como esta) y **celdas de código** (dentro de un recuadro, con un número entre corchetes a la izquierda).
- Para ejecutar una celda de código: haz clic en ella y pulsa **Mayús + Intro** (o el botón ▶ de la barra superior).
- Ejecuta las celdas **en orden, de arriba abajo**. Si algo se rompe: *Kernel → Restart Kernel and Run All Cells*.
- Guarda a menudo con **Ctrl + S** (**Cmd + S** en Mac). El cuaderno se guarda en la carpeta `cuadernos/` de tu ordenador.
- Las celdas marcadas con ✏️ son las que tienes que completar tú. Donde veas `...` o `# TODO`, escribe tu código.
- Algunas tareas se hacen en la **terminal** (apartados 2, 5 y 7): el cuaderno te dice qué comando ejecutar y dónde pegar la salida.
- Las preguntas de reflexión (🤔) se responden en la celda de texto que hay justo debajo de cada una.

## 0. Puesta en marcha

Ejecuta estas dos celdas cada vez que abras el cuaderno. La primera conecta con el NameNode de HDFS; la segunda muestra los archivos de tu área intermedia.

Si la primera da `Connection refused` o `safe mode`, HDFS todavía está arrancando: espera medio minuto y repítela.

In [28]:
from pathlib import Path
import io
import json

import pandas as pd
from hdfs import InsecureClient

AREA_INTERMEDIA = Path("/trabajo/area_intermedia")   # tu área intermedia (solo lectura)
DATALAKE = "/datalake"                              # raíz del data lake en HDFS
ZONAS = ["bruto", "limpio", "consumo"]

# Conexión con el NameNode a través de WebHDFS (puerto 9870)
hdfs = InsecureClient("http://namenode:9870", user="root")

print("pandas", pd.__version__)
print("Conectado con HDFS. Contenido de la raíz:", hdfs.list("/"))

pandas 2.2.3
Conectado con HDFS. Contenido de la raíz: []


In [29]:
archivos = sorted(p for p in AREA_INTERMEDIA.iterdir()
                  if p.is_file() and not p.name.startswith(".") and p.name != "LEEME.txt")

print(f"{len(archivos)} archivos en area_intermedia/")
for p in archivos:
    print(f"  {p.name:62} {p.stat().st_size / 1024**2:8.2f} MB")

21 archivos en area_intermedia/
  cadenaatlantica_acerca_de_2026-10-30.parquet                       0.00 MB
  cadenaatlantica_clientes_2026-10-30.parquet                        0.04 MB
  cadenaatlantica_habitaciones_2026-10-30.parquet                    0.01 MB
  cadenaatlantica_hoteles_2026-10-30.parquet                         0.01 MB
  cadenaatlantica_reservas-desde-2026-10-01_2026-10-30.parquet       0.01 MB
  cadenaatlantica_reservas_2026-10-30.parquet                        0.65 MB
  datoscanarias_hoteles_2026-10-29.csv                               0.15 MB
  datoscanarias_hoteles_2026-10-29.parquet                           0.06 MB
  datoscanarias_hoteles_2026-10-30.parquet                           0.06 MB
  datoscanarias_hoteles_2026-11-03.parquet                           0.06 MB
  datoscanarias_viviendas-vacacionales-original_2026-10-29.json     45.73 MB
  datoscanarias_viviendas-vacacionales-original_2026-10-30.json     45.72 MB
  datoscanarias_viviendas-vacacionales_2026-

### Herramientas para trabajar con HDFS desde Python

La variable `hdfs` es tu conexión. Estas son sus operaciones principales:

| Quiero… | Cómo |
| --- | --- |
| Ver qué hay en una carpeta | `hdfs.list("/datalake")` |
| Saber si algo existe | `hdfs.status(ruta, strict=False)` devuelve `None` si no existe |
| Tamaño, réplicas y bloque de un archivo | `hdfs.status(ruta)` → `["length"]`, `["replication"]`, `["blockSize"]` |
| Crear una carpeta (y las que falten por encima) | `hdfs.makedirs(ruta)` |
| Subir un archivo | `hdfs.upload(ruta_hdfs, ruta_local, overwrite=False)` |
| Leer un archivo | `with hdfs.read(ruta) as lector:` y dentro, por ejemplo, `pd.read_csv(lector)` |

## 1. Cómo guarda HDFS los archivos

Responde razonando, no con definiciones. Muestra los cálculos cuando los haya. Toma como base el tamaño de bloque por defecto, 128 MB.

**1.** Subes a HDFS un CSV de 300 MB, con un factor de replicación de 3. ¿En cuántos bloques se divide? ¿Cuánto espacio ocupa en total en el clúster?

✏️ *Tu respuesta:*
En principio, se dividiría en 3 bloques, siendo 9 bloques en total, 3 originales (300 / 128 = 2.x, que se redondea a 3) y 6 espejos por el factor de repliación.

**2.** Subes un archivo de 40 MB, como el CSV del ISTAC de la práctica 1. ¿Cuántos bloques ocupa? ¿Ocupa 128 MB en el disco del nodo?

✏️ *Tu respuesta:*
Ocupa sólo un bloque, y no, no ocupa 128MB porque 128 es el tamaño máximo del bloque, no un tamaño fijo que se rellena si no se llega.

**3.** Una fuente genera cada día 10 000 archivos JSON de 2 KB. ¿Los guardarías tal cual en HDFS? ¿Qué harías y por qué?

✏️ *Tu respuesta:*
No los guardaría tal cual en el HDFS porque vas a crear muchos bloques pequeños y sería un desperdicio.
Lo que intentaría es agrupar los JSON de tal manera que o ocupen la mayor parte posible del bloque o se organice de manera que tenga sentido.
Mi primera idea era guardarlo por semana, pero 20 * 7 = 140 > 128. 
A partir de aquí, veo dos opciones, o no generar un día a la semana (como un domingo, aunque no termino de ver esta opción)
o guardarlo en un formato más ligero como parquet y hacer guardar la información semanal en un bloque.
De esta manera, en vez de crear un 10.000 bloques al día, crearías uno a la semana.

**4.** Descubres un dato erróneo en un archivo de la zona en bruto. ¿Puedes corregirlo editando el archivo en HDFS? Aunque pudieras, ¿deberías hacerlo?

✏️ *Tu respuesta:*
No, HDFS no permite editar los bloques una vez creados, la única manera sería eliminando el bloque y volverlo a crear.
Y aunque pudieses, no deberías, ya que los errores también son útiles. Si hicieramos lo que acabo de decir de borrar
y volver a crear, perderíamos la procedencia del dato, con lo que no podríamos saber la fuente del fallo.

**5.** Tu empresa decide trasladar el *data lake* a un almacenamiento de objetos en la nube, como Amazon S3. Da una ventaja de ese cambio y una cosa que dejarías de ver o de controlar.

✏️ *Tu respuesta:*
Una ventaja de este cambio es la mejora en elasticidad. Con el modelo actual (HDFS), tienes un almacenamiento fijo, y al principio, estás gastando en almacenamiento
que no estás usando. Con almacenamiento en la nube, pagas por lo que usas, y depende de la cantidad de datos, puede ser un abaratamiento.
Una de las cosas que pierdes es la visibilidad de los bloques y el control de dónde va cada réplica.

## 2. Crear las zonas

Esta tarea se hace en la **terminal**, no aquí. Con el entorno en marcha, en la carpeta `SPS_UT1_P2_entorno`:

```bash
docker compose exec namenode hdfs dfs -mkdir -p /datalake/bruto /datalake/limpio /datalake/consumo
docker compose exec namenode hdfs dfs -ls /datalake
```

Después ejecuta la celda de abajo para comprobarlo desde Python, y copia en la celda de texto la salida del segundo comando de la terminal.

In [30]:
hdfs.list(DATALAKE)

['bruto', 'consumo', 'limpio']

**Salida de `hdfs dfs -ls /datalake`** ✏️
```
Pega aquí la salida de la terminal
Found 3 items
drwxr-xr-x   - root supergroup          0 2026-11-12 11:40 /datalake/bruto
drwxr-xr-x   - root supergroup          0 2026-11-12 11:40 /datalake/consumo
drwxr-xr-x   - root supergroup          0 2026-11-12 11:40 /datalake/limpio
```

## 3. Organizar la zona en bruto

Fíjate en los nombres de tus archivos (apartado 0): siguen la convención de la práctica 1, `fuente_tabla_AAAA-MM-DD.ext`. Decide una subcarpeta por fuente dentro de `/datalake/bruto/` y dónde guardarás los dos PDF y `registro_extracciones.csv`. Fíjate en que dos de las fuentes de la práctica 1 comparten prefijo: decide si van juntas o separadas.

Escribe la estructura en forma de árbol y justifica en 2 o 3 líneas por qué la organizas así (incluida tu decisión sobre las dos fuentes que comparten prefijo) y por qué conservas los nombres originales de los archivos.

**Estructura de la zona en bruto**

```
/datalake/bruto/
├── cadenaatlantica/
├──----- ...
├── datoscanarias_hoteles/
├──----- ...
├── datoscanarias_viviendas/
├──----- ...
├── istac/
├──----- ...
├── prueba/
├──----- ... (los de la modificación)
├── documentos/
├──----- .pdfs
├── registro_extracciones.csv
```

**Justificación:** ✏️ *Escribe aquí tu respuesta.*
Cadena Atlantica: 
Tiene muchos archivos con la misma fuente pero de distintas tablas.
cadenaatlantica_acerca_de_2026-10-30.parquet
cadenaatlantica_clientes_2026-10-30.parquet
cadenaatlantica_habitaciones_2026-10-30.parquet

entre otros

En la estructura actual, no renta hacer una subcarpeta por cada tabla porque sólo hay un archivo por tabla.
Si en algún punto hay, por ejemplo, 3 archivos por tabla de Cadena Atlántica, ahí ya habría que valorar hacer una subcarpeta por tabla
o separar por fecha, ya que en principio sólo puede haber un archivo de tabla por día.

Datos Canarias:
Separé hoteles de viviendas. El motivo principal es el formato

## 4. Cargar los archivos

Completa `subcarpeta()` y `cargar_en_bruto()`. Cada archivo de `area_intermedia/` tiene que ir a su subcarpeta de `/datalake/bruto/`:

- **sin modificarlo**: ni abrirlo con Pandas ni cambiar su formato;
- **sin sobrescribir** nada que ya exista en HDFS, como hiciste con `guardar()` en la modificación de la práctica 1.

Los PDF no siguen la convención de nombres de la práctica 1: decide cómo los trata tu función.

In [ ]:
def subcarpeta(nombre):
    """✏️ Devuelve la subcarpeta de /datalake/bruto/ donde va el archivo `nombre`."""
    # TODO: registro_extracciones.csv
    # TODO: los PDF
    # TODO: el resto, a partir de la convención fuente_tabla_AAAA-MM-DD.ext
    ...


def cargar_en_bruto(origen=AREA_INTERMEDIA, destino=f"{DATALAKE}/bruto"):
    """✏️ Sube cada archivo de `origen` a su subcarpeta de `destino`, sin modificarlo
    y sin sobrescribir. Devuelve dos listas: los archivos subidos y los que ya estaban."""
    subidos, ya_estaban = [], []
    for ruta in archivos:
        ruta_hdfs = f"{destino}/{subcarpeta(ruta.name)}/{ruta.name}"
        # TODO: si ruta_hdfs ya existe en HDFS, añádela a ya_estaban y pasa al siguiente archivo
        ...
        # TODO: sube el archivo tal cual, sin permitir que se sobrescriba
        ...
        subidos.append(ruta_hdfs)

    print(f"✔ Subidos: {len(subidos)} · ya estaban: {len(ya_estaban)}")
    for r in ya_estaban:
        print("  ya existía:", r)
    return subidos, ya_estaban

Ejecuta la carga una primera vez:

In [ ]:
subidos, ya_estaban = cargar_en_bruto()

Y una segunda vez. No debe sobrescribir nada: tiene que avisar de qué archivos ya estaban.

In [ ]:
subidos, ya_estaban = cargar_en_bruto()

## 5. Reconstruir el *data lake*

Tu *data lake* solo existe en este ordenador. Completa la celda de abajo para que, en una sola ejecución, **cree las zonas si no existen y cargue todo el área intermedia**. Si se ejecuta dos veces seguidas, no debe dar error ni duplicar archivos.

Pruébala desde cero:

1. En la terminal: `docker compose down -v` y después `docker compose up -d`. Tu *data lake* queda vacío.
2. Aquí: ejecuta el apartado 0, las definiciones del apartado 4 y después **solo** esta celda.
3. Ejecútala una segunda vez.

In [ ]:
# ✏️ Reconstruir el data lake
# TODO: crea las tres zonas si no existen
...
# TODO: carga toda el área intermedia en la zona en bruto
...

## 6. Comprobar la carga

Ejecuta la comprobación: muestra cada archivo con su tamaño en tu ordenador y en HDFS. Todas las líneas deben salir ✔ coincide. Si alguna sale ✘, revisa la carga de ese archivo. Los PDF solo se pueden comprobar así: no se pueden leer con Pandas.

In [ ]:
def tamano_hdfs(ruta_hdfs):
    estado = hdfs.status(ruta_hdfs, strict=False)
    return None if estado is None else estado["length"]

filas = []
for ruta in archivos:
    ruta_hdfs = f"{DATALAKE}/bruto/{subcarpeta(ruta.name)}/{ruta.name}"
    local, remoto = ruta.stat().st_size, tamano_hdfs(ruta_hdfs)
    filas.append({
        "ruta_hdfs": ruta_hdfs,
        "mb_local": round(local / 1024**2, 2),
        "mb_hdfs": None if remoto is None else round(remoto / 1024**2, 2),
        "comprobacion": "✔ coincide" if remoto == local else "✘ revisa",
    })
pd.DataFrame(filas)

### Leer desde HDFS ✏️

Lee desde HDFS **un archivo de cada formato** (CSV, Parquet y JSON) y comprueba que tiene las mismas filas y columnas que la copia de tu ordenador. El CSV está resuelto como ejemplo.

- Parquet: el lector necesita el archivo entero en memoria → `pd.read_parquet(io.BytesIO(lector.read()))`.
- JSON: el original no es una tabla → `json.load(lector)`, y compara la clave `result` con la tabla que guardaste en la práctica 1.

In [ ]:
# Ejemplo: un CSV
csv_local = next(p for p in archivos if p.suffix == ".csv" and p.name != "registro_extracciones.csv")
ruta_csv = f"{DATALAKE}/bruto/{subcarpeta(csv_local.name)}/{csv_local.name}"
with hdfs.read(ruta_csv) as lector:
    df_hdfs = pd.read_csv(lector, dtype=str, keep_default_na=False)
df_local = pd.read_csv(csv_local, dtype=str, keep_default_na=False)
print(csv_local.name, "· HDFS", df_hdfs.shape, "· local", df_local.shape,
      "✔" if df_hdfs.shape == df_local.shape else "✘")

In [ ]:
# ✏️ Un Parquet
parquet_local = ...   # TODO: elige uno de tus archivos .parquet
ruta_parquet = ...    # TODO: su ruta en HDFS
with hdfs.read(ruta_parquet) as lector:
    df_hdfs = ...     # TODO
df_local = ...        # TODO
print(parquet_local.name, "· HDFS", df_hdfs.shape, "· local", df_local.shape,
      "✔" if df_hdfs.shape == df_local.shape else "✘")

In [ ]:
# ✏️ El JSON original
json_local = ...      # TODO: tu archivo .json
ruta_json = ...       # TODO: su ruta en HDFS
with hdfs.read(ruta_json) as lector:
    datos = ...       # TODO
print(json_local.name, "· registros en HDFS:", len(datos["result"]))
# TODO: compara ese número con las filas de la tabla de viviendas que guardaste en la práctica 1

## 7. Observar las réplicas

En la **terminal**, lista una de tus subcarpetas (cambia `<tu_subcarpeta>` por la tuya):

```bash
docker compose exec namenode hdfs dfs -ls /datalake/bruto/<tu_subcarpeta>
```

La segunda columna de la salida es el **factor de replicación** de cada archivo. Copia la salida aquí debajo.

**Salida de `hdfs dfs -ls`** ✏️

```
Pega aquí la salida de la terminal
```

Para la pregunta sobre bloques del apartado 8, esta celda muestra el tamaño, las réplicas y el tamaño de bloque de tu archivo más grande:

In [ ]:
mayor = max(archivos, key=lambda p: p.stat().st_size)
estado = hdfs.status(f"{DATALAKE}/bruto/{subcarpeta(mayor.name)}/{mayor.name}")
print(mayor.name)
print(f"  tamaño: {estado['length'] / 1024**2:,.2f} MB · réplicas: {estado['replication']} "
      f"· tamaño de bloque: {estado['blockSize'] / 1024**2:,.0f} MB")

## 8. Preguntas de reflexión 🤔

Responde en la celda de texto que hay debajo de cada pregunta: de 3 a 5 líneas, con tus palabras y apoyándote en lo que has visto al ejecutar.

**🤔 1.** Has guardado los datos en el lago **sin limpiarlos**. ¿Por qué un *data lake* lo permite y un *data warehouse* no? Usa los términos *schema-on-read* y *schema-on-write*.

✏️ *Tu respuesta:*

**🤔 2.** En esta práctica no has sacado ningún resultado de los datos. ¿Qué valor tiene entonces haberlos guardado en el lago? ¿Qué características de HDFS lo hacen una tecnología eficiente para ese objetivo, y cuándo dejaría de serlo?

✏️ *Tu respuesta:*

**🤔 3.** Has guardado dos PDF en el mismo lago que tus tablas. ¿Podrías guardarlos en un *data warehouse*? ¿Qué haría falta para sacarles valor más adelante?

✏️ *Tu respuesta:*

**🤔 4.** En el apartado 7, el factor de replicación es 1. ¿Por qué? ¿Qué valor verías en un clúster real y qué ganarías con él?

✏️ *Tu respuesta:*

**🤔 5.** Elige el archivo más grande de tu zona en bruto. ¿En cuántos bloques lo ha dividido HDFS? ¿Qué cambiaría si fuera diez veces mayor?

✏️ *Tu respuesta:*

**🤔 6.** Las zonas son solo carpetas. ¿Qué impide que alguien guarde por error datos sin limpiar en la zona de consumo?

✏️ *Tu respuesta:*

**🤔 7.** ¿Por qué la zona en bruto **no debe modificarse nunca**? Relaciónalo con tu celda de reconstrucción.

✏️ *Tu respuesta:*

**🤔 8.** En la pregunta 7 de la práctica 1 pensaste qué cambiarías si los archivos tuvieran que guardarse en otro sitio. Ahora que lo has hecho, ¿qué ha cambiado realmente en tu código y qué se ha mantenido igual?

✏️ *Tu respuesta:*

---
### Resumen

Has llevado todo lo que guardaste en el área intermedia a la **zona en bruto** de un *data lake* en HDFS, **sin modificarlo ni sobrescribirlo**: tablas en CSV y Parquet, un JSON original, el registro de procedencia y dos documentos PDF. Has organizado el lago **por zonas** para que no se convierta en un pantano de datos y has comprobado que puedes **reconstruirlo** en cualquier ordenador a partir del área intermedia.